# EAP and EAP-IG workflow smoke test

This notebook exercises the EAP and EAP-IG Q&A workflows with a real Hugging Face causal LM and a reduced subset of the repo datasets. All test-specific constants live in the next cell.

In [ ]:
# Test-specific constants. Edit only this cell to change test cost or coverage.
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
MODEL_DTYPE = None
RANDOM_SEED = 0

DATA_FRACTION = 0.01
MAX_EXAMPLES_PER_DATASET = 1
BATCH_SIZE = 1

LAYER_COMPONENTS = [[0, "z"], [0, "mlp_hidden"]]
EAP_IG_STEPS = [1]
QUADRATURE = "riemann-midpoint"
COMPUTE_GRADIENT_AT = "clean"

OPTION_KEYS = ["(A)", "(B)"]
PROMPT_SUFFIX = "("
SYSTEM_PROMPT = "Recommend an approach for the given scenario. Reply only with (A) or (B)"
PROMPT_TEMPLATE = "{}\n\n{}\n{}"

TEST_ROOT = "results/workflow_smoke_tests/eap_eap_ig"
CONFIG_DIRNAME = "configs"
DATA_DIRNAME = "data"
EAP_IG_RESULTS_DIRNAME = "eap_ig_results"
EAP_RESULTS_DIRNAME = "eap_results"
TOP_COMPONENTS_DIRNAME = "top_components"
SELECTED_NODES_DIRNAME = "selected_nodes"
CLEAR_PREVIOUS_OUTPUTS = True

TOP_N = 8
SELECTION_LIMIT = 8
COMPUTE_COMPLETENESS = False
SAVE_TO_GCP = True
GCP_PROJECT_ID = None
GCS_BUCKET_NAME = None
GCS_TEST_PREFIX = None

In [ ]:
from __future__ import annotations

import json
import math
import shutil
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import yaml

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root.")

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from temporal_manifolds.eap_ig.eap_ig_qanda_pipeline import run_eap, run_eap_ig
from temporal_manifolds.utils.gcs_upload import _build_gcs_client
from temporal_manifolds.utils.eap_ig_artifacts import build_top_components, write_selected_nodes
from temporal_manifolds.utils.mech_interp_toolkit.utils import load_model_tokenizer_config, set_global_seed

In [ ]:
run_timestamp = datetime.now().strftime("%H:%M %d/%m/%y")
gcs_safe_timestamp = run_timestamp.replace("/", "-")
gcs_test_prefix = GCS_TEST_PREFIX or f"Test_{gcs_safe_timestamp}"

test_root = REPO_ROOT / TEST_ROOT
if CLEAR_PREVIOUS_OUTPUTS and test_root.exists():
    resolved = test_root.resolve()
    allowed_root = (REPO_ROOT / "results" / "workflow_smoke_tests").resolve()
    if allowed_root not in resolved.parents and resolved != allowed_root:
        raise RuntimeError(f"Refusing to delete unexpected path: {resolved}")
    shutil.rmtree(test_root)

config_dir = test_root / CONFIG_DIRNAME
data_dir = test_root / DATA_DIRNAME
eap_ig_results_dir = test_root / EAP_IG_RESULTS_DIRNAME
eap_results_dir = test_root / EAP_RESULTS_DIRNAME

for path in (config_dir, data_dir, eap_ig_results_dir, eap_results_dir):
    path.mkdir(parents=True, exist_ok=True)

print(f"Smoke-test artifacts will be written under: {test_root.relative_to(REPO_ROOT)}")
if SAVE_TO_GCP:
    missing_gcs_config = [
        name
        for name, value in (
            ("GCP_PROJECT_ID", GCP_PROJECT_ID),
            ("GCS_BUCKET_NAME", GCS_BUCKET_NAME),
        )
        if not value
    ]
    if missing_gcs_config:
        raise ValueError(f"Missing required GCS config variables: {', '.join(missing_gcs_config)}")
    print(f"Generated files will be uploaded under GCS prefix: {gcs_test_prefix}")
    print(f"Human timestamp: {run_timestamp}")

In [ ]:
source_files = {
    "explicit": REPO_ROOT / "data" / "binary_choice_questions" / "temporal_scope_explicit_expanded_500_debiased.json",
    "implicit": REPO_ROOT / "data" / "binary_choice_questions" / "temporal_scope_implicit_expanded_500_debiased.json",
}


def write_reduced_dataset(case_name: str, source_path: Path) -> Path:
    with source_path.open("r", encoding="utf-8") as f:
        payload = json.load(f)

    pairs = payload["pairs"]
    subset_size = max(1, min(MAX_EXAMPLES_PER_DATASET, math.ceil(len(pairs) * DATA_FRACTION)))
    reduced_payload = {
        "metadata": {
            **payload.get("metadata", {}),
            "smoke_test_source": str(source_path.relative_to(REPO_ROOT)),
            "smoke_test_n_pairs": subset_size,
        },
        "pairs": pairs[:subset_size],
    }

    output_path = data_dir / f"{case_name}_subset.json"
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(reduced_payload, f, indent=2)
    return output_path


reduced_data_files = {
    case_name: write_reduced_dataset(case_name, source_path)
    for case_name, source_path in source_files.items()
}
reduced_data_files

In [ ]:
def write_qanda_config(case_name: str, data_file: Path) -> Path:
    config = {
        "setup": {
            "model": MODEL_NAME,
            "seed": RANDOM_SEED,
            "batch_size": BATCH_SIZE,
            "dtype": MODEL_DTYPE,
            "granularity": "fine",
            "layer_components": LAYER_COMPONENTS,
            "quadrature": QUADRATURE,
        },
        "paths": {
            "data_loc": str(data_dir),
            "save_loc": f"./results/{case_name}/A",
        },
        "input": {
            "data_file": data_file.name,
            "template": PROMPT_TEMPLATE,
            "option_keys": OPTION_KEYS,
            "prompt_suffix": PROMPT_SUFFIX,
            "horizon": ["ST", "LT"],
        },
        "output": {
            "filename": f"A_{case_name}",
            "gcp_project_id": GCP_PROJECT_ID,
            "gcs_bucket_name": GCS_BUCKET_NAME,
            "gcs_prefix": gcs_test_prefix,
        },
        "parameters": {
            "system_prompt": SYSTEM_PROMPT,
            "steps": EAP_IG_STEPS,
            "metric_type": "option-logit",
        },
    }

    config_path = config_dir / f"{case_name}.yaml"
    with config_path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, sort_keys=False)
    return config_path


config_paths = {
    case_name: write_qanda_config(case_name, data_file)
    for case_name, data_file in reduced_data_files.items()
}
config_paths

In [ ]:
set_global_seed(RANDOM_SEED)
model, tokenizer, hf_config = load_model_tokenizer_config(
    MODEL_NAME,
    dtype=MODEL_DTYPE,
    suffix=PROMPT_SUFFIX,
    system_prompt=SYSTEM_PROMPT,
    attn_type="eager",
)

print(f"Loaded {MODEL_NAME} with {hf_config.num_hidden_layers} layers on {next(model.parameters()).device}")

In [ ]:
for case_name, config_path in config_paths.items():
    print(f"Running EAP-IG for {case_name}...")
    model, tokenizer = run_eap_ig(
        config_path,
        model=model,
        tokenizer=tokenizer,
        save_to_gcp=False,
        results_root=eap_ig_results_dir,
    )

for case_name, config_path in config_paths.items():
    print(f"Running EAP for {case_name}...")
    model, tokenizer = run_eap(
        config_path,
        model=model,
        tokenizer=tokenizer,
        save_to_gcp=False,
        results_root=eap_results_dir,
        compute_gradient_at=COMPUTE_GRADIENT_AT,
    )

In [ ]:
actual_examples_by_case = {}
for case_name, data_file in reduced_data_files.items():
    with data_file.open("r", encoding="utf-8") as f:
        actual_examples_by_case[case_name] = len(json.load(f)["pairs"])


def assert_raw_outputs(results_dir: Path, expected_method: str) -> list[Path]:
    files = sorted(results_dir.glob("*/*/*.npz"))
    expected_total = sum(
        2 * 2 * math.ceil(n_examples / BATCH_SIZE)
        for n_examples in actual_examples_by_case.values()
    )
    assert len(files) == expected_total, (len(files), expected_total, results_dir)

    with np.load(files[0], allow_pickle=False) as data:
        keys = set(data.files)
        assert "metadata__attribution_method" in keys
        assert data["metadata__attribution_method"].item() == expected_method
        assert any(key.endswith("__clean_logits") for key in keys)
        assert any("__z__0" in key for key in keys), sorted(keys)
        assert any("__mlp_hidden__0" in key for key in keys), sorted(keys)
    return files


eap_ig_npz = assert_raw_outputs(eap_ig_results_dir, "eap_ig")
eap_npz = assert_raw_outputs(eap_results_dir, "eap")
print(f"Validated {len(eap_ig_npz)} EAP-IG NPZ files and {len(eap_npz)} EAP NPZ files.")

In [ ]:
def build_and_validate_downstream_artifacts(results_dir: Path, artifact_label: str) -> dict[str, Path]:
    artifact_root = test_root / artifact_label
    top_components_dir = artifact_root / TOP_COMPONENTS_DIRNAME
    selected_nodes_dir = artifact_root / SELECTED_NODES_DIRNAME
    figures_dir = artifact_root / "figures"

    top_pickles, figures = build_top_components(
        results_dir=results_dir,
        top_components_dir=top_components_dir,
        completeness_figures_dir=figures_dir,
        top_n=TOP_N,
        compute_completeness=COMPUTE_COMPLETENESS,
    )
    selected_pickle, selected_json = write_selected_nodes(
        top_components_dir=top_components_dir,
        selected_nodes_dir=selected_nodes_dir,
        top_n=TOP_N,
        selection_limit=SELECTION_LIMIT,
        artifact_label=artifact_label,
    )

    for path in [*top_pickles.values(), selected_pickle, selected_json]:
        assert path.exists(), path
    if not COMPUTE_COMPLETENESS:
        assert figures == {}

    return {
        "explicit_top_components": top_pickles["explicit"],
        "implicit_top_components": top_pickles["implicit"],
        "selected_nodes": selected_pickle,
        "selected_nodes_json": selected_json,
    }


eap_ig_artifacts = build_and_validate_downstream_artifacts(eap_ig_results_dir, "eap_ig")
eap_artifacts = build_and_validate_downstream_artifacts(eap_results_dir, "eap")

print("EAP-IG artifacts:")
for name, path in eap_ig_artifacts.items():
    print(f"  {name}: {path.relative_to(REPO_ROOT)}")
print("EAP artifacts:")
for name, path in eap_artifacts.items():
    print(f"  {name}: {path.relative_to(REPO_ROOT)}")

In [ ]:
def upload_generated_files_to_gcp(root: Path, prefix: str) -> list[Path]:
    files = sorted(path for path in root.rglob("*") if path.is_file())
    if not SAVE_TO_GCP:
        return files

    bucket = _build_gcs_client(GCP_PROJECT_ID).bucket(GCS_BUCKET_NAME)
    resolved_prefix = prefix.strip("/")
    for path in files:
        object_name = path.relative_to(root).as_posix()
        if resolved_prefix:
            object_name = f"{resolved_prefix}/{object_name}"
        bucket.blob(object_name).upload_from_filename(str(path))
    return files


generated_files = upload_generated_files_to_gcp(test_root, gcs_test_prefix)
if SAVE_TO_GCP:
    print(f"Uploaded {len(generated_files)} generated files under GCS prefix: {gcs_test_prefix}")
else:
    print(f"Generated {len(generated_files)} files locally; GCS upload disabled.")

In [ ]:
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("EAP and EAP-IG workflow smoke test completed successfully.")